# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All entities (record sets, fields, and columns) are referenced **by their `@id`** to ensure semantic traceability.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:")
print(metadata.name)
print("\nDataset description:")
print(metadata.description)
print("\nCollection timeframe:")
print(metadata.dataCollectionTimeframe)
print("\nData biases:")
print(metadata.dataBiases)


## 2. Data Overview
Review available record sets, fields, and their IDs using Croissant entities.

### Show Record Sets and Their Fields

In Croissant, `@id` uniquely identifies each entity. We'll enumerate the record sets and their fields, referencing their `@id`.

In [ ]:
# The FAIR^2 dataset contains multiple record sets. Let's enumerate them and display their @id and fields.
record_sets = dataset.record_sets()

if not record_sets:
    print("No record sets found in metadata. If remote, try `dataset.reload()` or check schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {rs.description if hasattr(rs,'description') else 'N/A'}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id} | Name: {field.name}")
        print("\n---\n")

### Show Example Records from Each Record Set

Using the `@id` of each record set, we show several sample records. Replace `<id_of_the_records_set>` with actual record set `@id`.

In [ ]:
# Show examples for each record set, using their @id.
example_count = 3

for rs in record_sets:
    print(f"Example records for RecordSet @id: {rs.id}")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        if i >= example_count:
            break
        pprint.pprint(record)
    print("\n---\n")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing record set and field `@id`s as shown above.

In [ ]:
dataframes = {}

# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for RecordSet @id: {record_set_id}")
    print(df.columns.tolist())
    display(df.head(2))

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalization, and grouping. Use entity `@id` for all fields.

### Example: Filter and Normalize a Numeric Field in Clinical Features Record Set

- Suppose one record set tabulates clinical features, with numeric field `age at diagnosis`.
- We'll filter records where age > 20, normalize age, and group by a categorical field (e.g. presence of hirsutism).

In [ ]:
# Select a RecordSet that contains clinical features.
clinical_rs_id = None
for rs in record_sets:
    if rs.name.lower().startswith("clinical") or "Table 1" in rs.name:
        clinical_rs_id = rs.id
        break

if clinical_rs_id and clinical_rs_id in dataframes:
    clinical_df = dataframes[clinical_rs_id]
    # Find a numeric field by its @id (e.g. 'age at diagnosis' field @id)
    numeric_field_id = None
    group_field_id = None
    for field in [f for f in record_sets if f.id == clinical_rs_id][0].fields:
        if 'age' in field.name.lower():
            numeric_field_id = field.id
        if 'hirsutism' in field.name.lower():
            group_field_id = field.id

    if numeric_field_id:
        print(f"Numeric field @id: {numeric_field_id}")
        print(f"Group field @id: {group_field_id}")
        # Filter
        threshold = 20
        filtered_df = clinical_df[clinical_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field matching 'age' found in clinical features record set.")
else:
    print("Clinical features record set not found or not loaded.")

## 5. Visualization

Visualize data distributions or relationships between fields. All columns referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Example: Visualize distribution of normalized age at diagnosis
if clinical_rs_id and clinical_rs_id in dataframes and numeric_field_id:
    filtered_df = dataframes[clinical_rs_id][dataframes[clinical_rs_id][numeric_field_id] > 20]
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    plt.figure(figsize=(6,4))
    plt.hist(filtered_df[f"{numeric_field_id}_normalized"], bins=5, color='skyblue')
    plt.title(f"Distribution of normalized field: {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.grid(True)
    plt.show()
else:
    print("No clinical features, numeric field, or records available for visualization.")

## 6. Conclusion

- This notebook demonstrated loading and exploring a FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id` per the Croissant schema.
- Data exploration shows clear structure: clinical, laboratory, imaging, and genetic record sets.
- Example filtering, normalization, and grouping on clinical features are possible using dataset semantics.
- Limitations: small sample size, all patients female, restricted to single medical center—caution for generalization.
- See Croissant schema for further analysis and reproducibility in rare disease research contexts.
